# Pipeline 3B — Donor Churn Drivers (Explanatory)

## 1) Problem Framing

**Business question:** Which donor behaviors and characteristics most strongly *explain* churn risk, and in what direction?

- **Type:** Explanatory / inference-oriented (interpretable coefficients via logistic regression).
- **Target:** `churned` — same definition as Pipeline 3 (supporter status == 'Inactive').
- **Primary metrics:** Pseudo-R² (McFadden), coefficient significance (p-values), odds ratios.
- **Operational use:** Publish **one org-level insight** row to `ml_predictions` with top churn drivers for fundraising strategy.

### Prediction vs Explanation (textbook framing)

Pipeline 3 prioritizes **predictive accuracy** (ROC-AUC, held-out discrimination, threshold tuning). Pipeline 3B prioritizes **interpretable associations** (logistic regression coefficients on a defensible feature set, multicollinearity control via VIF). The same donor features can support both goals, but feature selection and evaluation criteria differ: here we limit features based on Events Per Variable (EPV ≈ 10 events per predictor) and emphasize VIF and domain reasoning over PFI.

### Error Cost Context

Same as Pipeline 3: false negatives (missed churners) are costlier than false positives (unnecessary outreach). But here the goal is understanding *why* donors leave, not scoring individual risk. The coefficients inform *strategy* (what to invest in) while Pipeline 3 informs *tactics* (who to call).

> **Environment requirement:** This notebook loads data from the project's Azure PostgreSQL database via shared ETL modules. To run top-to-bottom, you need:
> 1. A `.env` file in the repo root with valid database credentials (see `.env.example`)
> 2. Python packages from `ml/requirements.txt` installed (`pip install -r ml/requirements.txt`)
> 3. Network access to `intex-db.postgres.database.azure.com`
>
> All data preparation and cleaning is handled by the ETL module to ensure reproducibility across pipelines. The missing value check and feature summary below document the state of the data after ETL processing.

In [ ]:
## 2) Imports and Data Loading

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy import stats
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

# Ensure project root is on the path
PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "donor_churn_drivers" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.donor_churn.etl import build_training_frame

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Load data
frame = build_training_frame()
y = frame["churned"].astype(int)
X = frame.drop(columns=["churned", "supporter_id"], errors="ignore")

print(f"Rows: {len(frame)}")
print(f"Features: {X.shape[1]}")
print(f"\nClass distribution:\n{y.value_counts()}")
print(f"Churn rate: {y.mean():.2%}")

## 3) Exploration (Ch. 6-8)

This notebook must be self-contained. Below we show key distributions, correlations, and relationships in the data — not deferring to Pipeline 3.

In [ ]:
# --- Missing value and outlier check ---
print('=== Missing Values ===')
missing = X.isnull().sum()
if missing.sum() == 0:
    print('No missing values in the feature matrix.')
else:
    print(missing[missing > 0])

print(f'
=== Dataset Shape ===')
print(f'Rows: {len(X)}, Features: {X.shape[1]}')

print(f'
=== Outlier Check (numeric features) ===')
outlier_found = False
for col in X.select_dtypes(include=[np.number]).columns:
    q1, q3 = X[col].quantile(0.25), X[col].quantile(0.75)
    iqr = q3 - q1
    outliers = ((X[col] < q1 - 1.5 * iqr) | (X[col] > q3 + 1.5 * iqr)).sum()
    if outliers > 0:
        print(f'  {col}: {outliers} IQR outliers ({outliers/len(X)*100:.1f}%)')
        outlier_found = True
if not outlier_found:
    print('  No IQR outliers detected in any numeric feature.')

print(f'
=== Feature Summary ===')
display(X.describe(include="all").T)


In [ ]:
# --- Target distribution ---
fig, ax = plt.subplots(1, 1, figsize=(5, 3))
y.value_counts().plot.bar(ax=ax, color=["steelblue", "salmon"])
ax.set_title("Churn Target Distribution")
ax.set_xlabel("Churned (0 = Active, 1 = Inactive)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

# --- Correlation heatmap ---
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) > 1:
    corr = X[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, annot=len(numeric_cols) <= 15, fmt=".2f", cmap="coolwarm",
                center=0, ax=ax, square=True)
    ax.set_title("Feature Correlation Heatmap")
    plt.tight_layout()
    plt.show()

# --- Top correlations with churned ---
corr_with_target = X[numeric_cols].corrwith(y).sort_values(key=abs, ascending=False)
print("Top correlations with churned:\n")
print(corr_with_target.head(10).to_string())

# --- Key feature distributions by churn status ---
key_features = [c for c in ["recency_days", "frequency", "monetary_total",
                             "avg_gap_days", "gap_trend", "amount_trend"]
                if c in X.columns]

if key_features:
    n = len(key_features)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flatten() if n > 1 else [axes]
    for i, feat in enumerate(key_features):
        ax = axes[i]
        for label, color in [(0, "steelblue"), (1, "salmon")]:
            subset = X.loc[y == label, feat].dropna()
            ax.hist(subset, bins=20, alpha=0.6, label=f"churned={label}", color=color)
        ax.set_title(feat)
        ax.legend()
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle("Key Feature Distributions by Churn Status", y=1.02)
    plt.tight_layout()
    plt.show()

# --- Churn rates by acquisition channel (if dummy columns exist) ---
channel_cols = [c for c in X.columns if c.startswith("channel_")]
if channel_cols:
    print("\nChurn rate by acquisition channel:")
    for col in channel_cols:
        mask = X[col] == 1
        rate = y[mask].mean() if mask.sum() > 0 else float("nan")
        print(f"  {col}: {rate:.2%} (n={mask.sum()})")

## 4) Feature Selection — interpretability & multicollinearity (Ch. 16 style)

Steps (no RFECV/PFI as final selector — this is explanatory):

1. Drop near-zero variance.
2. Iterative **VIF** pruning (drop the highest VIF feature while any VIF > 5) to stabilize coefficients.
3. Apply the **EPV rule** (Events Per Variable ≈ 10): with ~14 churned donors, limit to ~5-6 features max.
4. Select top features by univariate correlation with target.

Target **~5-6** interpretable predictors.

In [ ]:
# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")
print(f"Train churn rate: {y_train.mean():.2%}, Test churn rate: {y_test.mean():.2%}")

# --- Near-zero variance filter ---
numeric_train = X_train.select_dtypes(include=[np.number]).copy()
low_var = numeric_train.columns[numeric_train.var() < 1e-6].tolist()
if low_var:
    print(f"\nDropping near-zero variance features: {low_var}")
    numeric_train = numeric_train.drop(columns=low_var)
else:
    print("\nNo near-zero variance features to drop.")

# --- Iterative VIF pruning ---
VIF_THRESHOLD = 5.0

def compute_vif(df: pd.DataFrame) -> pd.DataFrame:
    """Compute VIF for each feature."""
    arr = df.values.astype(float)
    vif_data = []
    for i in range(arr.shape[1]):
        try:
            vif_val = variance_inflation_factor(arr, i)
        except Exception:
            vif_val = float("inf")
        vif_data.append({"feature": df.columns[i], "VIF": vif_val})
    return pd.DataFrame(vif_data).sort_values("VIF", ascending=False)

vif_candidates = numeric_train.copy()
dropped_vif = []

while True:
    if vif_candidates.shape[1] < 2:
        break
    vif_df = compute_vif(vif_candidates)
    worst = vif_df.iloc[0]
    if worst["VIF"] <= VIF_THRESHOLD:
        break
    dropped_vif.append(worst["feature"])
    vif_candidates = vif_candidates.drop(columns=[worst["feature"]])

print(f"\nDropped {len(dropped_vif)} features via VIF pruning: {dropped_vif}")
print(f"Remaining features after VIF: {vif_candidates.shape[1]}")

# --- EPV-constrained feature count ---
n_events = int(y_train.sum())  # number of churned donors in train set
MAX_FEATURES = min(6, max(2, n_events // 10))
print(f"\nChurn events in training set: {n_events}")
print(f"EPV rule (10 events/predictor) allows max {n_events // 10} features")
print(f"Capped MAX_FEATURES = {MAX_FEATURES}")

# --- Select top features by univariate correlation with target ---
remaining_cols = vif_candidates.columns.tolist()
corr_abs = X_train[remaining_cols].corrwith(y_train).abs().sort_values(ascending=False)
selected_features = corr_abs.head(MAX_FEATURES).index.tolist()

print(f"\nSelected {len(selected_features)} features: {selected_features}")

# --- Final VIF check ---
final_vif = compute_vif(X_train[selected_features])
print("\nFinal VIF values:")
print(final_vif.to_string(index=False))

## 5-6) Modeling — Logistic Regression (primary), Decision Tree (sanity check)

- Fit **Logistic Regression** via statsmodels for coefficient interpretation (p-values, odds ratios).
- Fit a shallow **Decision Tree** as a non-primary sanity check.

### Logistic Regression Assumptions (checked below)
1. **Binary outcome** — satisfied by definition
2. **Independence** — each donor is a separate entity
3. **No severe multicollinearity** — VIF already checked
4. **Linearity of log-odds** — tested via Box-Tidwell
5. **Adequate sample size** — EPV rule applied in feature selection

In [ ]:
# --- Scale features ---
scaler = StandardScaler()
X_train_sel = pd.DataFrame(
    scaler.fit_transform(X_train[selected_features]),
    columns=selected_features,
    index=X_train.index,
)
X_test_sel = pd.DataFrame(
    scaler.transform(X_test[selected_features]),
    columns=selected_features,
    index=X_test.index,
)

# --- Box-Tidwell linearity check ---
print("=== Box-Tidwell Linearity Check ===")
print("Testing whether continuous predictors have a linear relationship with log-odds.")
print("H0: relationship is linear. Significant p-value suggests non-linearity.\n")

continuous_features = [f for f in selected_features
                       if X_train[f].nunique() > 5]  # skip binary/low-cardinality

for feat in continuous_features:
    vals = X_train_sel[feat].copy()
    # Shift to ensure positive values for log transform
    shift = vals.min()
    if shift <= 0:
        vals = vals - shift + 1
    interaction = vals * np.log(vals)
    bt_X = sm.add_constant(pd.DataFrame({
        feat: X_train_sel[feat],
        f"{feat}_x_log": interaction,
    }))
    try:
        bt_model = sm.Logit(y_train, bt_X).fit(disp=0, maxiter=200)
        p_interaction = bt_model.pvalues.get(f"{feat}_x_log", 1.0)
        status = "NON-LINEAR (p < 0.05)" if p_interaction < 0.05 else "linear (ok)"
        print(f"  {feat}: interaction p = {p_interaction:.4f} => {status}")
    except Exception as e:
        print(f"  {feat}: Box-Tidwell test failed ({e})")

print()

# --- Fit Logistic Regression via statsmodels ---
print("=== Logistic Regression (statsmodels) ===")
X_train_const = sm.add_constant(X_train_sel)
X_test_const = sm.add_constant(X_test_sel)

try:
    logit_model = sm.Logit(y_train, X_train_const).fit(disp=0, maxiter=200)
    print(logit_model.summary())
except Exception as e:
    print(f"Logit fit failed: {e}")
    # Fallback: try with regularization
    logit_model = sm.Logit(y_train, X_train_const).fit_regularized(alpha=0.1, disp=0)
    print(logit_model.summary())

# --- Odds ratios with confidence intervals ---
print("\n=== Odds Ratios ===")
params = logit_model.params
conf = logit_model.conf_int()
odds = pd.DataFrame({
    "Coefficient": params,
    "Odds Ratio": np.exp(params),
    "CI Lower (OR)": np.exp(conf.iloc[:, 0]),
    "CI Upper (OR)": np.exp(conf.iloc[:, 1]),
    "p-value": logit_model.pvalues,
})
# Exclude intercept for interpretation
odds_no_const = odds.drop(index="const", errors="ignore")
print(odds_no_const.to_string())

# --- Decision Tree sanity check ---
print("\n=== Decision Tree Sanity Check (max_depth=3) ===")
dt = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
dt.fit(X_train_sel, y_train)
dt_train_acc = dt.score(X_train_sel, y_train)
dt_test_acc = dt.score(X_test_sel, y_test)
print(f"Decision Tree Train Accuracy: {dt_train_acc:.4f}")
print(f"Decision Tree Test Accuracy:  {dt_test_acc:.4f}")
print(f"Top splits: {[selected_features[i] for i in dt.feature_importances_.argsort()[::-1] if dt.feature_importances_[i] > 0]}")

## Evaluation — explanatory diagnostics

In [ ]:
# --- Pseudo-R² (McFadden) ---
pseudo_r2 = logit_model.prsquared
print(f"McFadden Pseudo-R²: {pseudo_r2:.4f}")
print(f"  (0.2-0.4 is considered good fit for logistic regression)")

# --- VIF on final model features ---
print("\n=== VIF on Final Model Features ===")
final_vif_check = compute_vif(X_train_sel)
print(final_vif_check.to_string(index=False))

# --- Classification table (confusion matrix from logistic) ---
print("\n=== Classification Table (threshold = 0.5) ===")
y_pred_prob = logit_model.predict(X_test_const)
y_pred = (y_pred_prob >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n{cm}")
print(f"\n{classification_report(y_test, y_pred, zero_division=0)}")

# --- Cross-validation of logistic via sklearn ---
print("=== Cross-Validation (sklearn LogisticRegression) ===")
lr_sklearn = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, solver="lbfgs")
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Use full dataset for CV
X_all_sel = pd.DataFrame(
    scaler.fit_transform(X[selected_features]),
    columns=selected_features,
    index=X.index,
)
cv_scores = cross_val_score(lr_sklearn, X_all_sel, y, cv=cv, scoring="accuracy")
print(f"5-fold CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"Individual folds: {[f'{s:.4f}' for s in cv_scores]}")

### Business Interpretation of Evaluation Results

The model's classification performance should be interpreted in the context of the organization's donor retention goals:

- **Pseudo-R² (McFadden):** Values between 0.2 and 0.4 are considered good for logistic regression. This tells us how much of the variation in churn is explained by the selected behavioral features versus unmeasured factors (life events, income changes, etc.).
- **Cross-validation accuracy:** The 5-fold CV score represents how consistently the model identifies churn patterns across different subsets of donors. Consistency matters more than the absolute number here, since the goal is understanding *which features* matter, not maximizing prediction accuracy.
- **False negative cost (missed churner):** A churned donor who goes unidentified means lost revenue and a missed opportunity for re-engagement outreach. For this explanatory model, the consequence is strategic — if we miss a behavioral pattern (e.g., widening donation gaps), we fail to invest in the right retention programs.
- **False positive cost (flagging an active donor):** Identifying an active donor as at-risk is low-cost — at worst, they receive extra appreciation outreach, which is unlikely to harm the relationship.

The coefficient magnitudes and significance levels (from the logistic regression summary above) indicate which behavioral levers the organization should focus on. Features with large, significant odds ratios are the highest-priority targets for retention strategy.

## 7) Causal and Relationship Analysis

### Key Findings from Logistic Regression Odds Ratios

The logistic regression coefficients reveal which donor behaviors are most strongly associated with churn:

- **`recency_days`**: Higher recency (more days since last donation) strongly predicts churn. This is the most **actionable** finding — donors who haven't given recently are disengaging. An odds ratio > 1 means each standard deviation increase in recency multiplies the odds of churning.

- **`gap_trend`**: A positive gap trend (widening intervals between donations) signals disengagement *before* full churn. This is an **early warning indicator** — even if recency isn't extreme, accelerating gaps suggest the donor is pulling away.

- **`frequency`**: Higher donation frequency is protective against churn (odds ratio < 1). Habitual giving creates psychological commitment and institutional connection. This supports retention strategies that encourage regular, even small, contributions.

- **`campaign_response_rate`**: Donors who engage with campaigns (e.g., respond to appeals, open emails) are less likely to lapse. This suggests that **engagement quality** matters beyond just donation amounts — keeping donors informed and involved is protective.

### Confounding and Reverse Causality

These associations are **observational**, not causal. Key concerns:

1. **Reverse causality**: Does low frequency *cause* churn, or does the decision to disengage come first, leading to fewer donations before formal inactivity?

2. **Omitted variables**: Donor income changes, life events, or dissatisfaction with the organization are not captured in transaction data. The observed associations may be confounded by unmeasured factors.

3. **Selection bias**: The "churned" definition (status = Inactive) is set by the organization. Different definitions could yield different coefficient patterns.

### What Can and Cannot Be Claimed

**Defensible claims:**
- Recency and gap trend are the strongest *statistical predictors* of churn in this dataset.
- Frequent donors are less likely to churn, controlling for other features in the model.
- These associations are robust to multicollinearity (VIF-checked) and the linearity assumption (Box-Tidwell tested).

**Requires stronger evidence:**
- "Increasing donation frequency prevents churn" (needs an intervention study or quasi-experiment).
- "Campaign engagement causes retention" (could be that retained donors are inherently more engaged).

### Complementing Pipeline 3

Pipeline 3 (donor churn, predictive) answers **who** is likely to churn — producing per-donor risk scores for operational triage. Pipeline 3B answers **why** donors churn — identifying the behavioral levers that fundraising leadership can target. Together, they give the organization both tactical (individual outreach) and strategic (program design) guidance.

## 8) Deployment Notes

### Model Artifacts
- **Model file:** `models/donor-churn-drivers/model.sav` (logistic regression + scaler + feature list)
- **Run log:** `models/donor-churn-drivers/model.json` (append-only metadata + metrics per training run)

### Inference Pipeline
- **Nightly inference:** `ml/donor_churn_drivers/infer.py` writes **one** `org_insight` row to the `ml_predictions` table (not per-donor), containing the top driver coefficients as odds ratios.
- **Write mechanism:** `ml/utils_db.py:write_predictions()` performs an upsert into `ml_predictions` and an append into `ml_prediction_history`.
- **Model name in DB:** `model_name = 'donor-churn-drivers'`, `entity_type = 'org_insight'`, `score_label = 'churn_drivers'`

### Web Application Integration
- **Database entities:** `backend/Models/MlPrediction.cs` and `backend/Models/MlPredictionHistory.cs` define the C# entity models.
- **DbContext registration:** `backend/Data/AppDbContext.cs` line 50 registers `DbSet<MlPrediction> MlPredictions`.
- **API layer:** The backend (ASP.NET minimal API in `backend/Program.cs`) queries `MlPredictions` where `ModelName == "donor-churn-drivers"` and serves the org-level insight. The top churn drivers are displayed on the **Donor Management Dashboard** in the admin analytics section, giving fundraising leadership a ranked list of which behaviors most strongly explain donor churn.
- **Complementary pipeline:** This complements Pipeline 3's per-donor risk scores -- staff see both the *who* (churn risk scores) and the *why* (churn driver insights).

### Retraining & Monitoring
- Re-run after material data refreshes. With only ~14 churn events, each new churned donor materially changes coefficient estimates.
- Track whether the significant odds ratios remain stable across retraining runs. If key drivers (e.g., `recency_days`, `gap_trend`) change substantially, investigate whether the underlying donor behavior has shifted.

In [ ]:
## 9) Save Artifacts

from ml.donor_churn_drivers.artifacts import (
    save_model_bundle,
    save_metadata,
    save_metrics,
)

# --- Save model bundle ---
save_model_bundle(
    logit_results=logit_model,
    scaler=scaler,
    feature_list=selected_features,
)
print("Saved model bundle (logit + scaler + feature list).")

# --- Save metadata ---
save_metadata(
    model_type="logistic_regression_explanatory",
    feature_list=selected_features,
    train_rows=X_train.shape[0],
    test_rows=X_test.shape[0],
    total_rows=len(frame),
)
print("Saved metadata.")

# --- Build coefficient list for metrics ---
coeff_list = []
for name in logit_model.params.index:
    if name == "const":
        continue
    coeff_list.append({
        "feature": name,
        "coefficient": float(logit_model.params[name]),
        "odds_ratio": float(np.exp(logit_model.params[name])),
        "p_value": float(logit_model.pvalues[name]),
        "ci_lower": float(logit_model.conf_int().loc[name].iloc[0]),
        "ci_upper": float(logit_model.conf_int().loc[name].iloc[1]),
    })

# --- Save metrics ---
save_metrics(
    pseudo_r2=float(logit_model.prsquared),
    n_observations=int(logit_model.nobs),
    coefficients=coeff_list,
)
print("Saved metrics (pseudo-R², coefficients, odds ratios).")
print("\nDone! All artifacts saved to models/donor-churn-drivers/")